# Lekse 08 - Multi-Agent Designmønster


## Oppsett


In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

%pip install agent-framework azure-ai-projects azure-identity --quiet

import os
import asyncio

from agent_framework import AgentResponseUpdate, WorkflowBuilder
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Hvorfor Multi-Agent Systemer?

Virkelige oppgaver som reiseplanlegging involverer mange forskjellige typer ekspertise — logistikk, lokal kunnskap, budsjettering og mer. En enkelt agent som prøver å håndtere alt blir raskt uhåndterlig.

Multi-agent systemer løser dette gjennom **spesialisering**: hver agent fokuserer på ett ekspertiseområde, og leverer resultater av høyere kvalitet enn en generalist. De forbedrer også **skalerbarhet** — du kan legge til nye agenter (f.eks. en flyspesialist, en restaurantkritiker) uten å skrive om den eksisterende arbeidsflyten. Agentene settes sammen gjennom en strukturert prosess, hvor kontekst sendes videre fra en til den neste.


## Lage spesialiserte agenter


In [ ]:
planner_agent = await provider.create_agent(
    name="TravelPlanner",
    instructions="You are a travel planning specialist. Create detailed trip itineraries based on the traveler's preferences. Include daily schedules, must-see attractions, and logistical tips.",
)

concierge_agent = await provider.create_agent(
    name="TravelConcierge",
    instructions="You are a travel concierge who reviews and enhances trip plans. Review the plan for completeness, add local insider tips, suggest restaurants, and identify potential issues. Provide your feedback in a constructive format.",
)

## Bygge en Sekvensiell Arbeidsflyt

`WorkflowBuilder` lar deg koble agenter inn i en rettet graf. Her lager vi en enkel to-trinns pipeline: **TravelPlanner** utarbeider reiseplanen, deretter gjennomgår og forbedrer **TravelConcierge** den.


In [ ]:
workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .build()

last_author = None
events = workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## Legge til flere agenter i arbeidsflyten

En av de største fordelene med multi-agent-mønsteret er hvor enkelt det er å utvide. Nedenfor legger vi til en **BudgetReviewer**-agent som sjekker planen mot reisendes budsjett, markerer elementer som kan presse kostnadene over grensen, og foreslår pengerbesparende alternativer. Arbeidsflyten kjører nå tre agenter i rekkefølge:

```
TravelPlanner → TravelConcierge → BudgetReviewer
```


In [ ]:
budget_agent = await provider.create_agent(
    name="BudgetReviewer",
    instructions="You are a budget-conscious travel advisor. Review the proposed trip plan and concierge enhancements against the traveler's stated budget. Estimate costs for flights, hotels, meals, and activities. Flag anything that risks exceeding the budget and suggest cost-saving alternatives while preserving the trip's quality.",
)

extended_workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .add_edge(concierge_agent, budget_agent) \
    .build()

last_author = None
events = extended_workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## Sammendrag

I denne leksjonen lærte du hvordan du:

1. **Oppretter spesialiserte agenter** — hver med en fokusert rolle (planlegging, concierge, budsjettgjennomgang).
2. **Kobler agenter sammen i en sekvensiell arbeidsflyt** ved å bruke `WorkflowBuilder` og `add_edge`.
3. **Streamer utdata** fra en fler-agent pipeline, og følger med på hvilken agent som snakker.
4. **Utvider en arbeidsflyt** ved å legge til nye agenter i kjeden uten å endre eksisterende.

Designmønsteret med flere agenter holder hver agent enkel, samtidig som det produserer rikere, mer grundig gjennomgåtte resultater enn hva en enkelt agent kunne oppnå alene.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Ansvarsfraskrivelse**:
Dette dokumentet er oversatt ved hjelp av AI-oversettelsestjenesten [Co-op Translator](https://github.com/Azure/co-op-translator). Selv om vi streber etter nøyaktighet, vennligst vær oppmerksom på at automatiserte oversettelser kan inneholde feil eller unøyaktigheter. Det opprinnelige dokumentet på dets morsmål bør anses som den autoritative kilden. For kritisk informasjon anbefales profesjonell menneskelig oversettelse. Vi er ikke ansvarlige for eventuelle misforståelser eller feiltolkninger som oppstår ved bruk av denne oversettelsen.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
